In [1]:
%pip install -q playwright
!playwright install chromium

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# SEMICONDUCTOR PATENT CORPUS BUILDER
# ============================================================
#
# Source:
#   NortheasternUniversity/big_patent
#
# We stream CPC sections:
#   H = Electricity
#   G = Physics
#
# We DO NOT download the complete BIGPATENT dataset.
#
# Output:
#
# semiconductor_patents/
# ├── semiconductor_manufacturing/
# ├── transistor_device_technology/
# ├── memory/
# ├── advanced_packaging/
# ├── power_semiconductors/
# ├── photonics/
# ├── ai_advanced_semiconductor/
# └── metadata.csv
#
# Each .txt contains:
#   - abstract
#   - detailed patent description
#
# ============================================================


# ------------------------------------------------------------
# 1. Install datasets if necessary
# ------------------------------------------------------------

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("datasets") is None:
    print("Installing Hugging Face datasets...")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "datasets"
    ])


# ------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------

import csv
import hashlib
import re
import time
from pathlib import Path

from datasets import load_dataset


# ------------------------------------------------------------
# 3. Output directory
# ------------------------------------------------------------

ROOT = Path.cwd() / "semiconductor_patents"

ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 4. Targets
#
# Change these if you want 1,000 later.
# ------------------------------------------------------------

TARGETS = {
    "semiconductor_manufacturing": 100,
    "transistor_device_technology": 100,
    "memory": 75,
    "advanced_packaging": 75,
    "power_semiconductors": 50,
    "photonics": 50,
    "ai_advanced_semiconductor": 50,
}

TOTAL_TARGET = sum(TARGETS.values())


# ------------------------------------------------------------
# 5. Search vocabulary
#
# These are deliberately technical terms rather than simply
# searching for "semiconductor".
# ------------------------------------------------------------

KEYWORDS = {

    "semiconductor_manufacturing": [

        "semiconductor fabrication",
        "semiconductor manufacturing",
        "wafer fabrication",
        "wafer processing",
        "wafer manufacturing",
        "lithography",
        "photolithography",
        "euv lithography",
        "extreme ultraviolet lithography",
        "chemical mechanical polishing",
        "chemical mechanical planarization",
        "plasma etching",
        "dry etching",
        "wet etching",
        "ion implantation",
        "semiconductor deposition",
        "thin film deposition",
        "atomic layer deposition",
        "physical vapor deposition",
        "chemical vapor deposition",
        "semiconductor process",
        "fabrication process",
        "semiconductor cleaning",
        "wafer cleaning",
        "semiconductor substrate",
        "silicon wafer",
    ],

    "transistor_device_technology": [

        "finfet",
        "fin-fet",
        "fin field effect transistor",
        "gate all around",
        "gate-all-around",
        "gaa transistor",
        "nanosheet transistor",
        "nanowire transistor",
        "mosfet",
        "mos transistor",
        "field effect transistor",
        "field-effect transistor",
        "cmos transistor",
        "cmos device",
        "transistor device",
        "semiconductor device",
        "gate structure",
        "gate electrode",
        "source drain",
        "source/drain",
        "threshold voltage",
        "channel region",
        "channel layer",
        "complementary metal oxide semiconductor",
    ],

    "memory": [

        "semiconductor memory",
        "memory cell",
        "memory array",
        "memory device",
        "dram",
        "dynamic random access memory",
        "sram",
        "static random access memory",
        "nand flash",
        "nand memory",
        "flash memory",
        "3d nand",
        "three-dimensional nand",
        "nonvolatile memory",
        "non-volatile memory",
        "resistive memory",
        "reram",
        "resistive random access memory",
        "phase change memory",
        "pcm memory",
        "memory transistor",
        "memory array",
        "memory controller",
    ],

    "advanced_packaging": [

        "advanced packaging",
        "semiconductor packaging",
        "semiconductor package",
        "chiplet",
        "chiplets",
        "2.5d packaging",
        "2.5-d packaging",
        "3d packaging",
        "3-d packaging",
        "three dimensional packaging",
        "3d integrated circuit",
        "3d ic",
        "interposer",
        "silicon interposer",
        "package substrate",
        "wafer level packaging",
        "wafer-level packaging",
        "fan-out packaging",
        "fan out packaging",
        "flip chip",
        "hybrid bonding",
        "die stacking",
        "stacked die",
        "through silicon via",
        "through-silicon via",
        "tsv",
    ],

    "power_semiconductors": [

        "power semiconductor",
        "power semiconductor device",
        "power device",
        "power transistor",
        "power mosfet",
        "power mos",
        "silicon carbide",
        "sic semiconductor",
        "sic mosfet",
        "sic power",
        "gallium nitride",
        "gan semiconductor",
        "gan transistor",
        "gan power",
        "wide bandgap semiconductor",
        "wide-bandgap semiconductor",
        "wide band gap semiconductor",
        "igbt",
        "insulated gate bipolar transistor",
        "power diode",
        "schottky diode",
        "vertical power device",
    ],

    "photonics": [

        "silicon photonics",
        "silicon photonic",
        "integrated photonics",
        "photonic integrated circuit",
        "photonic integrated",
        "optical semiconductor",
        "semiconductor optical",
        "optical interconnect",
        "optical transceiver",
        "photonic device",
        "photonic circuit",
        "optical modulator",
        "electro optic modulator",
        "electro-optic modulator",
        "waveguide semiconductor",
        "silicon waveguide",
        "photodetector",
        "semiconductor laser",
        "laser diode",
        "optical computing",
    ],

    "ai_advanced_semiconductor": [

        "ai accelerator",
        "artificial intelligence accelerator",
        "machine learning accelerator",
        "neural network accelerator",
        "neural processing unit",
        "npu",
        "tensor processing unit",
        "tpu",
        "deep learning accelerator",
        "inference accelerator",
        "ai processor",
        "machine learning processor",
        "neural processor",
        "neuromorphic processor",
        "neuromorphic computing",
        "compute in memory",
        "compute-in-memory",
        "processing in memory",
        "processing-in-memory",
        "accelerator chip",
        "artificial intelligence processor",
    ],
}


# ------------------------------------------------------------
# 6. Compile patterns once
# ------------------------------------------------------------

PATTERNS = {}

for category, keywords in KEYWORDS.items():

    patterns = []

    for keyword in keywords:

        patterns.append(
            re.compile(
                re.escape(keyword),
                re.IGNORECASE
            )
        )

    PATTERNS[category] = patterns


# ------------------------------------------------------------
# 7. Create output directories
# ------------------------------------------------------------

for category in TARGETS:

    (
        ROOT /
        category
    ).mkdir(
        parents=True,
        exist_ok=True
    )


# ------------------------------------------------------------
# 8. Existing corpus
#
# This allows safe restarting.
# ------------------------------------------------------------

counts = {}

for category in TARGETS:

    counts[category] = len(
        list(
            (
                ROOT /
                category
            ).glob("*.txt")
        )
    )


print("=" * 72)
print("EXISTING CORPUS")
print("=" * 72)

for category, target in TARGETS.items():

    print(
        f"{category:42s}"
        f"{counts[category]:4d}/{target}"
    )

print("-" * 72)

print(
    f"TOTAL: {sum(counts.values())}/{TOTAL_TARGET}"
)


# ------------------------------------------------------------
# 9. Metadata
# ------------------------------------------------------------

METADATA = ROOT / "metadata.csv"

metadata_rows = []

if METADATA.exists():

    with open(
        METADATA,
        "r",
        encoding="utf-8",
        newline=""
    ) as f:

        reader = csv.DictReader(f)

        metadata_rows = list(reader)


existing_ids = {
    row.get("document_id")
    for row in metadata_rows
    if row.get("document_id")
}


# ------------------------------------------------------------
# 10. Text normalisation
# ------------------------------------------------------------

def clean_text(text):

    if not text:
        return ""

    text = str(text)

    text = text.replace(
        "\x00",
        " "
    )

    # Preserve paragraph boundaries where possible.
    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# ------------------------------------------------------------
# 11. Score abstract
#
# Abstract-first screening dramatically reduces the amount of
# full description matching we need to do.
# ------------------------------------------------------------

def abstract_scores(abstract):

    scores = {}

    for category, patterns in PATTERNS.items():

        score = 0

        for pattern in patterns:

            if pattern.search(abstract):

                score += 1

        scores[category] = score

    return scores


# ------------------------------------------------------------
# 12. Score full document
# ------------------------------------------------------------

def full_scores(abstract, description):

    text = (
        abstract +
        "\n" +
        description
    )

    scores = {}

    for category, patterns in PATTERNS.items():

        score = 0

        for pattern in patterns:

            # Count up to 3 occurrences.
            matches = pattern.findall(text)

            score += min(
                len(matches),
                3
            )

        scores[category] = score

    return scores


# ------------------------------------------------------------
# 13. Choose category
# ------------------------------------------------------------

def choose_category(
    abstract,
    description
):

    # First inspect abstract.
    a_scores = abstract_scores(
        abstract
    )

    best_abstract_category = max(
        a_scores,
        key=a_scores.get
    )

    best_abstract_score = (
        a_scores[
            best_abstract_category
        ]
    )

    # If the abstract has a strong technical signal,
    # we can immediately use it.
    if best_abstract_score >= 2:

        return (
            best_abstract_category,
            a_scores
        )

    # Otherwise inspect the description.
    scores = full_scores(
        abstract,
        description
    )

    best_category = max(
        scores,
        key=scores.get
    )

    return (
        best_category,
        scores
    )


# ------------------------------------------------------------
# 14. Save patent
# ------------------------------------------------------------

def save_patent(
    category,
    abstract,
    description,
    cpc_section
):

    abstract = clean_text(
        abstract
    )

    description = clean_text(
        description
    )

    if len(description) < 500:

        return None

    combined = (
        abstract +
        "\n" +
        description
    )

    document_id = hashlib.sha256(
        combined.encode("utf-8")
    ).hexdigest()[:20]

    if document_id in existing_ids:

        return None

    filename = (
        f"patent_{document_id}.txt"
    )

    output_path = (
        ROOT /
        category /
        filename
    )

    text = (
        "PATENT DOCUMENT\n"
        "================\n\n"
        f"DOCUMENT ID: {document_id}\n"
        f"CPC SECTION: {cpc_section.upper()}\n"
        f"CATEGORY: {category}\n"
        "SOURCE: BIGPATENT\n\n"
        "ABSTRACT\n"
        "========\n\n"
        f"{abstract}\n\n"
        "DETAILED DESCRIPTION\n"
        "====================\n\n"
        f"{description}\n"
    )

    output_path.write_text(
        text,
        encoding="utf-8"
    )

    existing_ids.add(
        document_id
    )

    return {
        "document_id": document_id,
        "category": category,
        "filename": filename,
        "abstract_length": len(abstract),
        "description_length": len(description),
        "cpc_section": cpc_section,
    }


# ------------------------------------------------------------
# 15. Save metadata
# ------------------------------------------------------------

def write_metadata():

    fieldnames = [
        "document_id",
        "category",
        "filename",
        "abstract_length",
        "description_length",
        "cpc_section",
        "source",
    ]

    with open(
        METADATA,
        "w",
        encoding="utf-8",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        writer.writeheader()

        for row in metadata_rows:

            writer.writerow({
                "document_id":
                    row.get("document_id", ""),

                "category":
                    row.get("category", ""),

                "filename":
                    row.get("filename", ""),

                "abstract_length":
                    row.get("abstract_length", ""),

                "description_length":
                    row.get("description_length", ""),

                "cpc_section":
                    row.get("cpc_section", ""),

                "source":
                    row.get(
                        "source",
                        "NortheasternUniversity/big_patent"
                    ),
            })


# ------------------------------------------------------------
# 16. Main streaming loop
# ------------------------------------------------------------

documents_seen = 0
documents_saved = 0

START_TIME = time.time()


def corpus_complete():

    return all(
        counts[category] >= TARGETS[category]
        for category in TARGETS
    )


print("\n")
print("=" * 72)
print("STARTING STREAM")
print("=" * 72)

print(
    "Source: NortheasternUniversity/big_patent"
)

print(
    "Streaming CPC sections: H + G"
)

print(
    "No complete dataset download."
)

print()


# ------------------------------------------------------------
# H first
#
# H = Electricity
#
# This is the most useful broad CPC section for semiconductor
# technology.
# ------------------------------------------------------------

for cpc_section in ["h", "g"]:

    if corpus_complete():
        break

    print(
        "\n" +
        "=" * 72
    )

    print(
        f"STREAMING CPC SECTION: "
        f"{cpc_section.upper()}"
    )

    print(
        "=" * 72
    )

    dataset = load_dataset(
        "NortheasternUniversity/big_patent",
        cpc_section,
        split="train",
        streaming=True,
    )

    for row in dataset:

        documents_seen += 1

        abstract = clean_text(
            row.get(
                "abstract",
                ""
            )
        )

        description = clean_text(
            row.get(
                "description",
                ""
            )
        )

        # ----------------------------------------------------
        # Skip obviously unusable records.
        # ----------------------------------------------------

        if len(description) < 500:

            continue

        # ----------------------------------------------------
        # Determine category.
        # ----------------------------------------------------

        category, scores = choose_category(
            abstract,
            description
        )

        best_score = scores[
            category
        ]

        # ----------------------------------------------------
        # Require technical relevance.
        # ----------------------------------------------------

        if best_score < 2:

            continue

        # ----------------------------------------------------
        # Category already full?
        # ----------------------------------------------------

        if (
            counts[category]
            >=
            TARGETS[category]
        ):

            continue

        # ----------------------------------------------------
        # Save immediately.
        # ----------------------------------------------------

        result = save_patent(
            category=category,
            abstract=abstract,
            description=description,
            cpc_section=cpc_section,
        )

        if result is None:

            continue

        counts[category] += 1
        documents_saved += 1

        metadata_rows.append({
            **result,
            "source":
                "NortheasternUniversity/big_patent"
        })

        # ----------------------------------------------------
        # Save metadata after every document.
        # ----------------------------------------------------

        write_metadata()

        # ----------------------------------------------------
        # Progress every document saved.
        # ----------------------------------------------------

        print(
            f"  + {category:38s} "
            f"{counts[category]:3d}/"
            f"{TARGETS[category]}"
        )

        # ----------------------------------------------------
        # Stop once complete.
        # ----------------------------------------------------

        if corpus_complete():

            break

        # ----------------------------------------------------
        # Progress heartbeat every 1,000 records.
        #
        # This is important: we can now see that the stream is
        # actually moving even when no patent has matched.
        # ----------------------------------------------------

        if documents_seen % 1000 == 0:

            elapsed = (
                time.time() -
                START_TIME
            )

            print(
                f"\n  scanned: "
                f"{documents_seen:,} | "
                f"saved: "
                f"{documents_saved:,} | "
                f"elapsed: "
                f"{elapsed / 60:.1f} min"
            )

            print(
                "  current counts:"
            )

            for category in TARGETS:

                print(
                    f"    {category:38s}"
                    f"{counts[category]:3d}/"
                    f"{TARGETS[category]}"
                )

            print()


# ------------------------------------------------------------
# 17. Final verification
# ------------------------------------------------------------

write_metadata()


print("\n")
print("=" * 72)
print("FINAL CORPUS")
print("=" * 72)

grand_total = 0

for category, target in TARGETS.items():

    directory = (
        ROOT /
        category
    )

    pdfs = list(
        directory.glob("*.txt")
    )

    count = len(pdfs)

    grand_total += count

    status = (
        "✓"
        if count >= target
        else "!"
    )

    print(
        f"{status} "
        f"{category:42s}"
        f"{count:4d}/{target}"
    )

print("-" * 72)

print(
    f"TOTAL: {grand_total}/{TOTAL_TARGET}"
)

print()

print(
    f"Corpus:\n{ROOT}"
)

print()

print(
    f"Metadata:\n{METADATA}"
)

print()

print(
    f"Records scanned: {documents_seen:,}"
)

print(
    f"Documents saved: {documents_saved:,}"
)


# ------------------------------------------------------------
# 18. Completion message
# ------------------------------------------------------------

if grand_total >= TOTAL_TARGET:

    print()
    print(
        "🎉 CORPUS COMPLETE"
    )

    print(
        f"Successfully collected "
        f"{grand_total} patent text documents."
    )

else:

    print()
    print(
        "⚠ CORPUS INCOMPLETE"
    )

    print(
        "The stream ended before all targets were reached."
    )

    print(
        "\nRemaining:"
    )

    for category, target in TARGETS.items():

        current = len(
            list(
                (
                    ROOT /
                    category
                ).glob("*.txt")
            )
        )

        if current < target:

            print(
                f"  {category}: "
                f"{target - current} remaining"
            )

EXISTING CORPUS
semiconductor_manufacturing                100/100
transistor_device_technology               100/100
memory                                      75/75
advanced_packaging                          11/75
power_semiconductors                        45/50
photonics                                   50/50
ai_advanced_semiconductor                   50/50
------------------------------------------------------------------------
TOTAL: 431/500


STARTING STREAM
Source: NortheasternUniversity/big_patent
Streaming CPC sections: H + G
No complete dataset download.


STREAMING CPC SECTION: H


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

  + advanced_packaging                      12/75
  + power_semiconductors                    46/50
  + power_semiconductors                    47/50
  + power_semiconductors                    48/50
  + power_semiconductors                    49/50
  + advanced_packaging                      13/75
  + advanced_packaging                      14/75
  + power_semiconductors                    50/50
  + advanced_packaging                      15/75
  + advanced_packaging                      16/75
  + advanced_packaging                      17/75
  + advanced_packaging                      18/75
  + advanced_packaging                      19/75
  + advanced_packaging                      20/75
  + advanced_packaging                      21/75
  + advanced_packaging                      22/75
  + advanced_packaging                      23/75
  + advanced_packaging                      24/75
  + advanced_packaging                      25/75
  + advanced_packaging                      26/75


In [6]:
# ------------------------------------------------------------
# 11. Random sample inspection — SAFE VERSION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("9. RANDOM DOCUMENT SAMPLES")
print("=" * 70)

random.seed(42)

# Use the categories actually present in the loaded documents
available_categories = sorted(
    set(d["category"] for d in documents)
)

print("\nCategories found in documents:")
for category in available_categories:
    count = sum(
        1 for d in documents
        if d["category"] == category
    )
    print(f"  {category}: {count}")

for category in TARGETS:

    category_docs = [
        d for d in documents
        if d["category"] == category
    ]

    # Don't crash if a category is missing
    if not category_docs:
        print("\n" + "-" * 70)
        print(f"⚠️ NO DOCUMENTS FOUND: {CATEGORY_LABELS.get(category, category)}")
        print("-" * 70)
        continue

    sample = random.choice(category_docs)

    text = sample["text"]

    # Extract abstract
    abstract_match = re.search(
        r"ABSTRACT\s*=+\s*(.*?)\s*DETAILED DESCRIPTION",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    if abstract_match:
        abstract = abstract_match.group(1).strip()
    else:
        abstract = "Abstract section not found."

    print("\n" + "-" * 70)
    print(CATEGORY_LABELS.get(category, category))
    print("-" * 70)

    print(f"File: {sample['filename']}")
    print(f"Words: {sample['words']:,}")

    print("\nAbstract preview:")
    print(abstract[:1000])

# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("10. FINAL VALIDATION")
print("=" * 70)

checks = {
    "500 metadata records": len(df) == 500,
    "500 readable documents": len(documents) == 500,
    "No missing files": len(missing_files) == 0,
    "No empty files": len(empty_files) == 0,
    "No read errors": len(read_errors) == 0,
    "No duplicate documents": len(duplicate_groups) == 0,
    "Unique document IDs": (
        "document_id" not in df.columns
        or df["document_id"].is_unique
    ),
}

for check, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'}  {check}"
    )

all_passed = all(checks.values())

print("\n" + "=" * 70)

if all_passed:
    print("🎉 CORPUS VALIDATION PASSED")
    print("The 500-document corpus is ready for chunking.")
else:
    print("⚠️ CORPUS VALIDATION FOUND ISSUES")
    print("Review the failures above before chunking.")

print("=" * 70)


9. RANDOM DOCUMENT SAMPLES

Categories found in documents:
  advanced_packaging: 64
  power_semiconductors: 5

----------------------------------------------------------------------
⚠️ NO DOCUMENTS FOUND: Semiconductor Manufacturing
----------------------------------------------------------------------

----------------------------------------------------------------------
⚠️ NO DOCUMENTS FOUND: Transistor / Device Technology
----------------------------------------------------------------------

----------------------------------------------------------------------
⚠️ NO DOCUMENTS FOUND: Memory
----------------------------------------------------------------------

----------------------------------------------------------------------
Advanced Packaging
----------------------------------------------------------------------
File: patent_5469465e8f806fc2b1fe.txt
Words: 10,685

Abstract preview:
An inertial sensor, comprises a detection element detecting an amount of a physical quantity

In [7]:
# ============================================================
# DIAGNOSE CORPUS FILE / METADATA MISMATCH
# ============================================================

from pathlib import Path
import pandas as pd

CORPUS_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents"
)

metadata_path = CORPUS_ROOT / "metadata.csv"

df_check = pd.read_csv(metadata_path)

print("=" * 70)
print("CORPUS DIAGNOSTIC")
print("=" * 70)

print(f"\nMetadata records: {len(df_check)}")

print("\nMetadata category counts:")
print(df_check["category"].value_counts().sort_index())

# ------------------------------------------------------------
# Count actual files directly from disk
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACTUAL FILES ON DISK")
print("=" * 70)

disk_counts = {}

for category in TARGETS:
    category_dir = CORPUS_ROOT / category

    if category_dir.exists():
        files = list(category_dir.glob("*.txt"))
    else:
        files = []

    disk_counts[category] = len(files)

    print(f"{category:40s} {len(files):4d}")

print("-" * 70)
print(f"{'TOTAL':40s} {sum(disk_counts.values()):4d}")

# ------------------------------------------------------------
# Compare metadata against disk
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("METADATA vs DISK")
print("=" * 70)

comparison = []

for category in TARGETS:

    metadata_count = int(
        (df_check["category"] == category).sum()
    )

    disk_count = disk_counts[category]

    comparison.append({
        "category": category,
        "metadata": metadata_count,
        "disk": disk_count,
        "difference": disk_count - metadata_count,
        "status": "PASS" if metadata_count == disk_count else "CHECK",
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

# ------------------------------------------------------------
# Check every metadata filename against disk
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("METADATA FILE REFERENCES")
print("=" * 70)

missing = []

for _, row in df_check.iterrows():

    expected_path = (
        CORPUS_ROOT
        / row["category"]
        / row["filename"]
    )

    if not expected_path.exists():
        missing.append(expected_path)

print(f"Metadata files: {len(df_check)}")
print(f"Missing from disk: {len(missing)}")

if missing:
    print("\nFirst missing files:")
    for path in missing[:20]:
        print(path)

# ------------------------------------------------------------
# Final diagnosis
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DIAGNOSIS")
print("=" * 70)

total_disk = sum(disk_counts.values())

if len(df_check) == 500 and total_disk == 500 and not missing:
    print("✅ THE 500-DOCUMENT CORPUS IS INTACT.")
    print("The previous validation was using stale/incomplete in-memory data.")
elif len(df_check) == 500 and total_disk < 500:
    print("⚠️ Metadata contains 500 records, but fewer than 500 files exist.")
else:
    print("⚠️ There is a genuine corpus mismatch.")

print("=" * 70)

CORPUS DIAGNOSTIC

Metadata records: 69

Metadata category counts:
category
advanced_packaging      64
power_semiconductors     5
Name: count, dtype: int64

ACTUAL FILES ON DISK
semiconductor_manufacturing               100
transistor_device_technology              100
memory                                     75
advanced_packaging                         75
power_semiconductors                       50
photonics                                  50
ai_advanced_semiconductor                  50
----------------------------------------------------------------------
TOTAL                                     500

METADATA vs DISK


,category,metadata,disk,difference,status
0,semiconductor_manufacturing,0,100,100,CHECK
1,transistor_device_technology,0,100,100,CHECK
2,memory,0,75,75,CHECK
3,advanced_packaging,64,75,11,CHECK
4,power_semiconductors,5,50,45,CHECK
5,photonics,0,50,50,CHECK
6,ai_advanced_semiconductor,0,50,50,CHECK



METADATA FILE REFERENCES
Metadata files: 69
Missing from disk: 0

DIAGNOSIS
⚠️ There is a genuine corpus mismatch.


In [15]:
# ============================================================
# REBUILD ENRICHED METADATA FROM THE 500 PATENT FILES
# ============================================================

from pathlib import Path
import csv
import re

CORPUS_ROOT = Path(
    "/Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents"
)

OUTPUT_FILE = CORPUS_ROOT / "metadata.csv"

CATEGORIES = [
    "semiconductor_manufacturing",
    "transistor_device_technology",
    "memory",
    "advanced_packaging",
    "power_semiconductors",
    "photonics",
    "ai_advanced_semiconductor",
]

records = []

print("=" * 80)
print("REBUILDING ENRICHED PATENT METADATA")
print("=" * 80)


# ============================================================
# HELPERS
# ============================================================

def extract_field(text, patterns):
    """
    Try several regex patterns and return the first match.
    """
    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE | re.MULTILINE
        )

        if match:
            value = match.group(1).strip()

            if value:
                return value

    return ""


def extract_section(text, start_patterns, end_patterns):
    """
    Extract a document section between headings.
    """

    start_match = None

    for pattern in start_patterns:
        start_match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE | re.MULTILINE
        )

        if start_match:
            break

    if not start_match:
        return ""

    start = start_match.end()

    remaining = text[start:]

    end_positions = []

    for pattern in end_patterns:

        match = re.search(
            pattern,
            remaining,
            flags=re.IGNORECASE | re.MULTILINE
        )

        if match:
            end_positions.append(match.start())

    if end_positions:
        end = min(end_positions)
        return remaining[:end].strip()

    return remaining.strip()


# ============================================================
# PROCESS FILES
# ============================================================

for category in CATEGORIES:

    category_dir = CORPUS_ROOT / category

    files = sorted(category_dir.glob("*.txt"))

    print(f"{category:40s}: {len(files):3d}")

    for path in files:

        text = path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        # ----------------------------------------------------
        # Existing BIGPATENT metadata
        # ----------------------------------------------------

        document_id = extract_field(
            text,
            [
                r"DOCUMENT ID:\s*(.+)",
            ],
        )

        if not document_id:
            document_id = path.stem.replace("patent_", "")


        cpc_section = extract_field(
            text,
            [
                r"CPC SECTION:\s*(.+)",
            ],
        )


        file_category = extract_field(
            text,
            [
                r"CATEGORY:\s*(.+)",
            ],
        )

        if not file_category:
            file_category = category


        source = extract_field(
            text,
            [
                r"SOURCE:\s*(.+)",
            ],
        )

        if not source:
            source = "BIGPATENT"


        # ----------------------------------------------------
        # Try to recover patent/publication identifiers
        # ----------------------------------------------------

        publication_number = extract_field(
            text,
            [
                r"PUBLICATION NUMBER:\s*(.+)",
                r"PUBLICATION NO\.?:\s*(.+)",
                r"PATENT PUBLICATION NUMBER:\s*(.+)",
                r"PATENT NUMBER:\s*(.+)",
                r"US PATENT(?: NO\.?| NUMBER)?:\s*(.+)",
            ],
        )


        # ----------------------------------------------------
        # Title
        # ----------------------------------------------------

        title = extract_field(
            text,
            [
                r"TITLE:\s*(.+)",
                r"TITLE OF INVENTION:\s*(.+)",
            ],
        )


        # ----------------------------------------------------
        # Abstract
        # ----------------------------------------------------

        abstract = extract_section(
            text,

            [
                r"^\s*ABSTRACT\s*=*\s*$",
                r"^\s*ABSTRACT\s*$",
            ],

            [
                r"^\s*DETAILED DESCRIPTION",
                r"^\s*DESCRIPTION",
                r"^\s*CLAIMS",
            ],
        )


        # ----------------------------------------------------
        # Claims
        # ----------------------------------------------------

        claims = extract_section(
            text,

            [
                r"^\s*CLAIMS\s*=*\s*$",
                r"^\s*CLAIMS\s*$",
            ],

            [
                r"^\s*REFERENCES CITED",
                r"^\s*REFERENCE",
                r"^\s*DRAWINGS",
                r"^\s*FIGURES",
            ],
        )


        # ----------------------------------------------------
        # Description
        # ----------------------------------------------------

        description = extract_section(
            text,

            [
                r"^\s*DETAILED DESCRIPTION\s*=*\s*$",
                r"^\s*DETAILED DESCRIPTION\s*$",
                r"^\s*DESCRIPTION\s*=*\s*$",
            ],

            [
                r"^\s*CLAIMS\s*=*\s*$",
                r"^\s*CLAIMS\s*$",
                r"^\s*REFERENCES CITED",
            ],
        )


        # ----------------------------------------------------
        # Inventors / assignee / dates
        # ----------------------------------------------------

        inventors = extract_field(
            text,
            [
                r"INVENTORS?:\s*(.+)",
                r"INVENTOR:\s*(.+)",
            ],
        )


        assignee = extract_field(
            text,
            [
                r"ASSIGNEE:\s*(.+)",
                r"CURRENT ASSIGNEE:\s*(.+)",
            ],
        )


        filing_date = extract_field(
            text,
            [
                r"FILING DATE:\s*(.+)",
            ],
        )


        publication_date = extract_field(
            text,
            [
                r"PUBLICATION DATE:\s*(.+)",
            ],
        )


        # ----------------------------------------------------
        # Basic statistics
        # ----------------------------------------------------

        word_count = len(
            re.findall(r"\b\w+\b", text)
        )

        character_count = len(text)


        # ----------------------------------------------------
        # Store record
        # ----------------------------------------------------

        records.append({

            "document_id": document_id,

            "publication_number": publication_number,

            "title": title,

            "abstract": abstract,

            "claims": claims,

            "description": description,

            "cpc_section": cpc_section,

            "inventors": inventors,

            "assignee": assignee,

            "filing_date": filing_date,

            "publication_date": publication_date,

            "category": category,

            "filename": path.name,

            "source": source,

            "word_count": word_count,

            "character_count": character_count,

        })


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("VALIDATING")
print("=" * 80)

print(f"\nTotal records: {len(records)}")


# ============================================================
# CATEGORY COUNTS
# ============================================================

for category in CATEGORIES:

    count = sum(
        1
        for r in records
        if r["category"] == category
    )

    print(
        f"{category:40s}: {count:3d}"
    )


# ============================================================
# DUPLICATES
# ============================================================

document_ids = [
    r["document_id"]
    for r in records
]

filenames = [
    r["filename"]
    for r in records
]

print(
    "\nDuplicate document IDs:",
    len(document_ids) - len(set(document_ids))
)

print(
    "Duplicate filenames:   ",
    len(filenames) - len(set(filenames))
)


# ============================================================
# FIELD COVERAGE
# ============================================================

print("\n" + "=" * 80)
print("FIELD COVERAGE")
print("=" * 80)

fields_to_check = [
    "publication_number",
    "title",
    "abstract",
    "claims",
    "description",
    "inventors",
    "assignee",
    "filing_date",
    "publication_date",
]

for field in fields_to_check:

    populated = sum(
        1
        for r in records
        if r[field]
    )

    print(
        f"{field:25s}: "
        f"{populated:3d} / {len(records):3d}"
    )


# ============================================================
# WRITE METADATA
# ============================================================

fieldnames = [
    "document_id",
    "publication_number",
    "title",
    "abstract",
    "claims",
    "description",
    "cpc_section",
    "inventors",
    "assignee",
    "filing_date",
    "publication_date",
    "category",
    "filename",
    "source",
    "word_count",
    "character_count",
]

with open(
    OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(records)


print("\nMetadata written to:")
print(OUTPUT_FILE)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

REBUILDING ENRICHED PATENT METADATA
semiconductor_manufacturing             : 100
transistor_device_technology            : 100
memory                                  :  75
advanced_packaging                      :  75
power_semiconductors                    :  50
photonics                               :  50
ai_advanced_semiconductor               :  50

VALIDATING

Total records: 500
semiconductor_manufacturing             : 100
transistor_device_technology            : 100
memory                                  :  75
advanced_packaging                      :  75
power_semiconductors                    :  50
photonics                               :  50
ai_advanced_semiconductor               :  50

Duplicate document IDs: 0
Duplicate filenames:    0

FIELD COVERAGE
publication_number       :   1 / 500
title                    :   0 / 500
abstract                 : 500 / 500
claims                   :   0 / 500
description              : 500 / 500
inventors                :   2 / 5

In [14]:
print(df.columns.tolist())
print()
display(df.head(3).T)

['document_id', 'category', 'filename', 'cpc_section', 'source', 'word_count', 'character_count', 'abstract']



,0,1,2
document_id,035ed7683a8cf2c3,08064dc5e6a41c78,095d694dbb617b65
category,semiconductor_manufacturing,semiconductor_manufacturing,semiconductor_manufacturing
filename,patent_035ed7683a8cf2c3.txt,patent_08064dc5e6a41c78.txt,patent_095d694dbb617b65.txt
cpc_section,NaN,NaN,NaN
source,BIGPATENT,BIGPATENT,BIGPATENT
word_count,3538,6450,12363
character_count,22987,38272,73608
abstract,A method for semiconductor device feature deve...,NaN,NaN


In [19]:
from pathlib import Path
from urllib.parse import urljoin
import requests
import re

PAGE_URL = "https://data.uspto.gov/bulkdata/datasets/ptgrxml"

print("=" * 80)
print("DOWNLOADING USPTO FRONTEND ASSETS — FIXED")
print("=" * 80)

html = requests.get(
    PAGE_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
).text

scripts = re.findall(
    r'<script[^>]+src="([^"]+)"',
    html,
    flags=re.I,
)

print(f"\nFound {len(scripts)} script tags:\n")

for s in scripts:
    url = urljoin(PAGE_URL, s)
    filename = url.split("/")[-1].split("?")[0]

    print(f"Downloading: {filename}")
    print(f"URL: {url}")

    try:
        r = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=60,
        )

        print(
            f"Status: {r.status_code} | "
            f"Bytes: {len(r.content):,}"
        )

        if (
            r.status_code == 200
            and filename.endswith(".js")
            and len(r.content) > 10_000
        ):
            Path(filename).write_bytes(r.content)
            print("SAVED")

        print()

    except Exception as e:
        print("ERROR:", repr(e))
        print()

print("=" * 80)
print("JAVASCRIPT FILES AVAILABLE")
print("=" * 80)

for path in sorted(Path(".").glob("*.js")):
    print(
        f"{path.name:45s} "
        f"{path.stat().st_size:12,} bytes"
    )

DOWNLOADING USPTO FRONTEND ASSETS — FIXED

Found 7 script tags:

Downloading: jquery-3.7.1.slim.min.js
URL: https://code.jquery.com/jquery-3.7.1.slim.min.js
Status: 200 | Bytes: 70,264
SAVED

Downloading: bootstrap.bundle.min.js
URL: https://cdn.jsdelivr.net/npm/bootstrap@4.5.3/dist/js/bootstrap.bundle.min.js
Status: 200 | Bytes: 84,152
SAVED

Downloading: challenge.js
URL: https://0dd6fc7fe1e2.edge.sdk.awswaf.com/0dd6fc7fe1e2/cee3ab3bc019/challenge.js
Status: 200 | Bytes: 695,178
SAVED

Downloading: dynamic-trans-menu-2.0.0.js
URL: https://components.uspto.gov/json/dynamic-trans-menu-2.0.0.js
Status: 200 | Bytes: 3,565

Downloading: polyfills-5CFQRCPP.js
URL: https://data.uspto.gov/bulkdata/datasets/polyfills-5CFQRCPP.js
Status: 200 | Bytes: 20,666
SAVED

Downloading: scripts-GYUJCTCT.js
URL: https://data.uspto.gov/bulkdata/datasets/scripts-GYUJCTCT.js
Status: 200 | Bytes: 20,666
SAVED

Downloading: main-U52U7DJC.js
URL: https://data.uspto.gov/bulkdata/datasets/main-U52U7DJC.js
Status

In [23]:
import requests
import re

URL = "https://data.uspto.gov/bulkdata/datasets/ptappclm"

print("=" * 80)
print("USPTO OCE PATENT CLAIMS DATASET")
print("=" * 80)
print("URL:", URL)

r = requests.get(
    URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60,
)

print("Status:", r.status_code)
print("Final URL:", r.url)
print("Content-Type:", r.headers.get("content-type"))
print("Bytes:", len(r.content))

text = r.text

print("\nSearching response for download/API references...\n")

patterns = [
    r'https?://[^"\']+',
    r'/api/[^"\']+',
    r'/download[^"\']+',
    r'[^"\']+\.csv',
    r'[^"\']+\.zip',
    r'[^"\']+\.dta',
]

found = set()

for pattern in patterns:
    for match in re.findall(pattern, text, flags=re.I):
        found.add(match)

for item in sorted(found):
    print(item[:1000])

USPTO OCE PATENT CLAIMS DATASET
URL: https://data.uspto.gov/bulkdata/datasets/ptappclm
Status: 200
Final URL: https://data.uspto.gov/bulkdata/datasets/ptappclm
Content-Type: text/html
Bytes: 20666

Searching response for download/API references...

https://0dd6fc7fe1e2.edge.sdk.awswaf.com/0dd6fc7fe1e2/cee3ab3bc019/challenge.js
https://cdn.jsdelivr.net/npm/bootstrap@4.5.3/dist/js/bootstrap.bundle.min.js
https://code.jquery.com/jquery-3.7.1.slim.min.js
https://components.uspto.gov/json/dynamic-trans-menu-2.0.0.js
https://fonts.gstatic.com
https://fonts.gstatic.com/s/materialsymbolsoutlined/v367/kJEhBvYX7BgnkSrUwT8OhrdQw4oELdPIeeII9v6oFsI.woff2) format(
https://zn3qmkzmofott7msc-uspto.gov1.siteintercept.qualtrics.com/SIE/?Q_ZID=ZN_3qMKzMOfOtT7MsC
